# 02 · Feature cleanup and random forest baseline

Two jobs:

1. **Feature selection** (step 3 of the QSAR workflow): drop constant and duplicated columns.
   Only exact redundancy goes; correlated-but-distinct features stay, because Model 1 uses
   ARD priors instead of manual selection.
2. **Baseline** (the reference every Bayesian model is measured against): random forest on
   each feature set, on both frozen splits.

The question this answers: **which feature set actually predicts flying** — Morgan fingerprints,
the lab's xenosite phase1 descriptors, my interpretable chemistry features, or a combination.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, matthews_corrcoef, log_loss,
                             brier_score_loss, average_precision_score)
from sklearn.calibration import calibration_curve

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = ROOT / "data" / "processed"

SEED = 20261124
ADDITIVE = "formate"        # primary; fluoride checked at the end
N_TREES = 500

## 1. Load and align

In [ ]:
cur = pd.read_csv(PROC / "wif_curated.csv")
chem = pd.read_csv(PROC / "features_chem.csv")
phase1 = pd.read_csv(PROC / "features_phase1.csv")
morgan = np.load(PROC / "morgan_2048_r2.npz", allow_pickle=True)["fp"]

for name, d in [("chem", chem), ("phase1", phase1)]:
    assert (d["compound_id"].to_numpy() == cur["compound_id"].to_numpy()).all(), name
assert len(morgan) == len(cur)

chem = chem.drop(columns=["compound_id"])
phase1 = phase1.drop(columns=["compound_id"])
print(f"{len(cur)} compounds | chem {chem.shape[1]} | phase1 {phase1.shape[1]} | morgan {morgan.shape[1]}")

## 2. Feature cleanup

Three kinds of redundancy, all checked against the data rather than assumed:

- **constant**: one value for every compound, carries nothing
- **duplicated inside a block**: `MolDesc__heavy_atoms` repeats `MolDesc__atoms`, because the
  lab's code counts atoms *after* hydrogens are added, so "heavy" is a misnomer here
- **duplicated across blocks**: RDKit and OpenBabel both give molecular weight and H-bond donors

Everything else stays, including pairs correlated at 0.95–0.99 (`MolMR` vs `MolDesc__MR`), which
measure related but not identical things.

In [ ]:
def drop_redundant(blocks: dict[str, pd.DataFrame], r_exact: float = 0.999) -> dict[str, pd.DataFrame]:
    """Drop constant columns, then one of each pair with |r| > r_exact. Later blocks lose the duplicate."""
    joined = pd.concat(blocks.values(), axis=1)
    const = [c for c in joined if joined[c].nunique() <= 1]

    keep = joined.drop(columns=const)
    corr = keep.corr(numeric_only=True).abs().to_numpy().copy()
    np.fill_diagonal(corr, 0.0)
    cols = keep.columns.to_list()
    dupes = {cols[j] for i in range(len(cols)) for j in range(i + 1, len(cols)) if corr[i, j] > r_exact}

    removed = set(const) | dupes
    print(f"constant    : {const}")
    print(f"duplicated  : {sorted(dupes)}")
    return {k: v.drop(columns=[c for c in v.columns if c in removed]) for k, v in blocks.items()}

chem, phase1 = drop_redundant({"chem": chem, "phase1": phase1}).values()
print(f"\nkept: chem {chem.shape[1]}, phase1 {phase1.shape[1]}")

## 3. Feature sets and evaluation

`morgan` is the baseline feature set named in the proposal. The combinations test whether the
lab's descriptors and the interpretable chemistry add anything on top of it.

In [ ]:
FEATURE_SETS = {
    "morgan":            morgan,
    "chem":              chem.to_numpy(float),
    "phase1":            phase1.to_numpy(float),
    "chem+phase1":       np.hstack([chem.to_numpy(float), phase1.to_numpy(float)]),
    "morgan+chem":       np.hstack([morgan, chem.to_numpy(float)]),
    "all":               np.hstack([morgan, chem.to_numpy(float), phase1.to_numpy(float)]),
}
{k: v.shape[1] for k, v in FEATURE_SETS.items()}

In [ ]:
def evaluate(X, y, split, n_trees=N_TREES, seed=SEED):
    """Fit on the training rows of a frozen split, score the held-out test rows."""
    tr, te = split == "train", split == "test"
    rf = RandomForestClassifier(n_estimators=n_trees, n_jobs=-1, random_state=seed,
                                min_samples_leaf=2, class_weight=None)
    rf.fit(X[tr], y[tr])
    p = rf.predict_proba(X[te])[:, 1]
    return {
        "AUC":      roc_auc_score(y[te], p),
        "AP":       average_precision_score(y[te], p),
        "MCC":      matthews_corrcoef(y[te], (p > 0.5).astype(int)),
        "log loss": log_loss(y[te], p),
        "Brier":    brier_score_loss(y[te], p),
    }, p, np.where(te)[0], rf

y = cur[f"flies_{ADDITIVE}"].to_numpy()
rows, preds, models = [], {}, {}
for split_name in ["split_random", "split_cluster"]:
    split = cur[split_name].to_numpy()
    for fs, X in FEATURE_SETS.items():
        m, p, idx, rf = evaluate(X, y, split)
        rows.append({"split": split_name.replace("split_", ""), "features": fs, **m})
        preds[(split_name, fs)] = (idx, p)
        models[(split_name, fs)] = rf
        print(f"{split_name.replace('split_',''):8s} {fs:14s} AUC {m['AUC']:.3f}  MCC {m['MCC']:.3f}")

results = pd.DataFrame(rows)

In [ ]:
pivot = results.pivot(index="features", columns="split", values=["AUC", "MCC", "log loss"]).round(3)
pivot = pivot.reindex(FEATURE_SETS.keys())
pivot

### Reading the table

The proposal quotes **AUC 0.89** from a quick random forest on the formate label; the `morgan`
row is the direct comparison. The `cluster` column is the harder test — whole clusters of similar
molecules are held out, so it estimates performance on genuinely new chemistry.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, split in zip(axes, ["random", "cluster"]):
    sub = results[results["split"] == split].set_index("features").reindex(FEATURE_SETS.keys())
    ax.barh(sub.index, sub["AUC"], color="#2a78d6", height=0.6)
    ax.set_xlim(0.5, 1.0); ax.axvline(0.89, ls="--", color="0.4", lw=1)
    ax.set_title(f"{split} split"); ax.set_xlabel("AUC (test)")
    ax.invert_yaxis()
axes[0].text(0.892, -0.45, "proposal 0.89", fontsize=8, color="0.4")
plt.tight_layout(); plt.show()

## 4. Are the probabilities honest?

A random forest votes; the vote share is not a calibrated probability. This plot is the baseline
the Bayesian models have to beat — a point on the diagonal means "when the model says 70%, it is
right 70% of the time".

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
for ax, split_name in zip(axes, ["split_random", "split_cluster"]):
    for fs in ["morgan", "chem", "all"]:
        idx, p = preds[(split_name, fs)]
        frac, mean = calibration_curve(y[idx], p, n_bins=10, strategy="quantile")
        ax.plot(mean, frac, "o-", ms=4, lw=1.4, label=fs)
    ax.plot([0, 1], [0, 1], "--", color="0.6", lw=1)
    ax.set_title(split_name.replace("split_", "") + " split")
    ax.set_xlabel("predicted probability")
axes[0].set_ylabel("observed fraction flying"); axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

## 5. Which chemistry does the model use?

Impurity importance from the `chem+phase1` forest. This is a sanity check, not an explanation:
it is biased towards high-cardinality features and says nothing about direction. Model 1 answers
the same question properly, with a sign and a credible interval on each effect.

In [ ]:
rf = models[("split_random", "chem+phase1")]
names = list(chem.columns) + list(phase1.columns)
imp = pd.Series(rf.feature_importances_, index=names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(6, 4.5))
top = imp.head(20)[::-1]
ax.barh(top.index, top.values, color="#2a78d6", height=0.65)
ax.set_xlabel("impurity importance"); ax.set_title("chem + phase1 forest, random split")
plt.tight_layout(); plt.show()

print("where the interpretable flags rank:")
for f in ["has_basic_N", "has_acid", "n_basic_N", "n_carboxylic_acid"]:
    print(f"  {f:20s} rank {list(imp.index).index(f) + 1:3d} / {len(imp)}   importance {imp[f]:.4f}")

## 6. The other additive

Ammonium fluoride has a much higher pass rate (78.6% vs 52.9%), so the task is more imbalanced
and the AUC is not directly comparable to the formate numbers above.

In [ ]:
y_f = cur["flies_fluoride"].to_numpy()
for fs in ["morgan", "chem", "all"]:
    m, *_ = evaluate(FEATURE_SETS[fs], y_f, cur["split_random"].to_numpy())
    print(f"fluoride  {fs:8s} AUC {m['AUC']:.3f}  MCC {m['MCC']:.3f}  (base rate {y_f.mean():.1%})")

## 7. Save the baseline numbers

In [ ]:
results.to_csv(PROC / "baseline_rf.csv", index=False)
print(results.round(3).to_string(index=False))